<a href="https://colab.research.google.com/github/clairewyh/SSU-Data-Dash-team-pineapple/blob/main/team_pineapple_Catboost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SSU Data Dash: quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/erwinzhang7/SSU-Data-Dash/blob/main/notebooks/quickstart.ipynb)

The same steps as `python run.py validate / report / predict / check`, one cell at a time.
Run the cells from top to bottom. Colab already has every package this needs.

In [ ]:

# SETUP: Clone Team Pineapple's GitHub repository.
# This downloads the team's code into Google Colab.

import os
import sys

TEAM_REPO = "https://github.com/clairewyh/SSU-Data-Dash-team-pineapple.git"
TEAM_FOLDER = "/content/SSU-Data-Dash-team-pineapple"

# Clone the repository if it isn't already downloaded.
if not os.path.exists(TEAM_FOLDER):
    !git clone -q {TEAM_REPO} {TEAM_FOLDER}

# Switch to the team project folder.
os.chdir(TEAM_FOLDER)

# Allow Python to find the project files.
if TEAM_FOLDER not in sys.path:
    sys.path.insert(0, TEAM_FOLDER)

print("Current project folder:", os.getcwd())
print("Python version:", sys.version.split()[0])
print("Setup complete!")


Current project folder: /content/SSU-Data-Dash-team-pineapple
Python version: 3.13.18
Setup complete!


## 1. Get the data

The files download from Hugging Face the first time and are cached for later runs. If you downloaded them by hand, set `DATA_DIR` in the next cell instead.

In [ ]:
# os.environ["DATA_DIR"] = "/path/to/folder"   # only if you downloaded the files yourself

import data
listings = data.load("listings")
print(f"{len(listings):,} sales, {listings['date_sold'].min():%B %Y} to {listings['date_sold'].max():%B %Y}")
listings.head()

listings.parquet: reconstructing file:   0%|          |  0.00B / 32.9MB            

listings.parquet: downloading bytes:           |  0.00B            

150,000 sales, January 2021 to May 2026


,ml_num,sold_price,list_price,date_listed,date_sold,days_on_market,house_type_name,house_style,bedrooms,washrooms,...,house_area,lot_front,lot_depth,municipality,community,latitude,longitude,external_avm_estimate,external_avm_estimate_date,remarks
0,SYN-0023247,364100.0,371200.0,2020-12-29,2021-01-01,3,Condo Apt,Apartment,1,1,...,622,NaN,NaN,Toronto,Sumacpark,43.602039,-79.261184,366200.0,2021-02-14,Looking For More Space In A Turnkey Condo? Thi...
1,SYN-0156988,599000.0,595800.0,2020-12-26,2021-01-01,6,Condo Apt,Apartment,2,1,...,649,NaN,NaN,Toronto,Hickorymills,43.695388,-79.510865,622800.0,2021-12-26,"Imagine a quiet evening on Sequoia Trail, wher..."
2,SYN-0087706,492900.0,462200.0,2020-12-22,2021-01-01,10,Condo Apt,Apartment,2,1,...,900-1099,NaN,NaN,North York,Hazelmills,43.740512,-79.386216,NaN,2020-12-03,"Discover this inviting 2-bedroom, 1-bath condo..."
3,SYN-0128154,532300.0,524900.0,2020-12-19,2021-01-01,13,Condo Apt,Apartment,1,1,...,876,NaN,NaN,Scarborough,Sumacbank,43.643774,-79.470180,573600.0,2021-02-22,Sumacbank presents this 1-bedroom condo on Mag...
4,SYN-0118596,564100.0,558300.0,2020-12-05,2021-01-01,27,Condo Apt,Apartment,1,1,...,514,NaN,NaN,Toronto,Ashworth,43.696172,-79.274137,513000.0,2021-02-05,"Morning coffee at the kitchen island, then a s..."


In [ ]:
off_market = data.load("off_market")
list_aware = data.load("list_aware")
print(f"{len(off_market):,} homes to value with no list price (off-market)")
print(f"{len(list_aware):,} homes to value that have a list price (list-aware)")
off_market.head()

In [ ]:
# A first look: the median sold price each month, by property type.
monthly = listings.groupby([listings["date_sold"].dt.to_period("M"), "house_type_name"])["sold_price"].median()
monthly.unstack().plot(figsize=(9, 4), ylabel="Median sold price ($)", xlabel="");

## 2. Validate

Each month from December 2025 to May 2026 is predicted by models trained only on the sales
before it, the same way the real test works. This takes about a minute.

In [ ]:
import run
run.validate()

2025-12: trained on 139,931 earlier sales, predicted 1,238 homes
2026-01: trained on 141,169 earlier sales, predicted 1,058 homes
2026-02: trained on 142,227 earlier sales, predicted 1,442 homes
2026-03: trained on 143,669 earlier sales, predicted 1,816 homes
2026-04: trained on 145,485 earlier sales, predicted 2,224 homes
2026-05: trained on 147,709 earlier sales, predicted 2,291 homes
Wrote outputs/validation_predictions.csv


In [ ]:
run.report()
from IPython.display import Image
Image("figures/error_by_price_range.png")

## 3. Improve the models

Edit **features.py** and **models.py**, then rerun the two cells above to see whether your
validation error goes down. In Colab, open them from the folder icon on the left
(`SSU-Data-Dash`), edit, and save with Ctrl+S (Cmd+S on a Mac).
Colab deletes your files when the session ends, so download anything you change.

### Experiment: CatBoost with advanced housing features

In [ ]:

# Install CatBoost, a model that can learn from numerical
# and categorical property information.

%pip install -q catboost


In [ ]:

%%writefile pineapple_catboost.py

# Team Pineapple: CatBoost experiment
# Purpose: Test a new algorithm using housing features
# that were useful in our previous experiments.

import numpy as np
import pandas as pd
from catboost import CatBoostRegressor


# Words from our team's previous feature experiments
GOOD_WORDS = [
    "redone", "turnkey", "updated", "newer",
    "modern", "fresh", "cared", "custom"
]

BAD_WORDS = [
    "needs", "potential", "opportunity",
    "bones", "original", "older", "vision"
]

# CatBoost will learn the price patterns associated
# with these categorical variables.
CATEGORIES = [
    "house_type_name", "house_style",
    "municipality", "community"
]


def make_features(homes, track):
    """Convert raw housing records into model inputs."""

    X = pd.DataFrame(index=homes.index)

    # 1. House size
    # Convert "900-1099" into its midpoint, 999.5.
    area = homes["house_area"].astype(str)
    parts = area.str.split("-", n=1, expand=True)
    low = pd.to_numeric(parts[0], errors="coerce")
    high = (
        pd.to_numeric(parts[1], errors="coerce")
        if 1 in parts.columns else low
    )
    X["square_feet"] = (low + high.fillna(low)) / 2

    # 2. Physical features
    for col in ["bedrooms", "washrooms",
                "parking_total", "garage"]:
        X[col] = pd.to_numeric(
            homes[col], errors="coerce"
        )

    # 3. Lot area
    X["lot_area"] = (
        pd.to_numeric(homes["lot_front"], errors="coerce")
        * pd.to_numeric(homes["lot_depth"], errors="coerce")
    )

    # 4. Property categories and location
    for col in CATEGORIES:
        X[col] = homes[col].fillna("Unknown").astype(str)

    X["latitude"] = homes["latitude"]
    X["longitude"] = homes["longitude"]

    # 5. Historical time information
    dates = pd.to_datetime(homes["date_listed"])
    X["months_since_2021"] = (
        dates - pd.Timestamp("2021-01-01")
    ).dt.days / 30.44

    X["month_of_year"] = dates.dt.month

    # 6. Listing description
    text = homes["remarks"].fillna("").str.lower()

    X["good_words"] = sum(
        text.str.contains(w, regex=False).astype(int)
        for w in GOOD_WORDS
    )
    X["bad_words"] = sum(
        text.str.contains(w, regex=False).astype(int)
        for w in BAD_WORDS
    )

    # New detailed property-condition indicators
    X["renovation"] = text.str.contains(
        r"renovat|remodel|upgrad",
        regex=True
    ).astype(int)

    X["finished_basement"] = text.str.contains(
        r"finished basement|basement apartment",
        regex=True
    ).astype(int)

    X["needs_work"] = text.str.contains(
        r"as.is|fixer.upper|needs work",
        regex=True
    ).astype(int)

    X["luxury"] = text.str.contains(
        r"luxury|high.end|premium finishes",
        regex=True
    ).astype(int)

    # 7. List-aware homes have a known asking price.
    if track == "list_aware":
        X["list_price"] = pd.to_numeric(
            homes["list_price"], errors="coerce"
        )

    return X


class CatBoostPriceModel:
    """Train one independent model for either track."""

    def __init__(self, track):
        self.track = track
        self.model = CatBoostRegressor(
            iterations=350,
            depth=6,
            learning_rate=0.05,
            loss_function="RMSE",
            random_seed=42,
            has_time=True,
            thread_count=2,
            verbose=False,
            allow_writing_files=False
        )

    def fit(self, train):
        # Fit only on records passed by the validation
        # pipeline. No hidden-test prices are accessed.
        train = train.sort_values(
            ["date_sold", "ml_num"]
        )

        X = make_features(train, self.track)

        if self.track == "list_aware":
            # Learn the logarithm of sold/list ratio.
            # Example: $700k sold / $680k list = 1.0294.
            y = np.log(
                train["sold_price"].to_numpy()
                / train["list_price"].to_numpy()
            )
        else:
            # Learn logarithm of final selling price.
            y = np.log(train["sold_price"].to_numpy())

        self.model.fit(
            X,
            y,
            cat_features=CATEGORIES
        )
        return self

    def predict(self, homes):
        X = make_features(homes, self.track)

        log_prediction = self.model.predict(X)

        if self.track == "list_aware":
            return (
                homes["list_price"].to_numpy(dtype=float)
                * np.exp(log_prediction)
            )

        return np.exp(log_prediction)


Writing pineapple_catboost.py


In [ ]:

# CATBOOST EXPERIMENT: Walk-forward validation
# This tests our model on historical sales from
# December 2025 to May 2026.

import run
import models
import pineapple_catboost

# Ensure our experiment is still selected.
models.make_model = lambda track: (
    pineapple_catboost.CatBoostPriceModel(track)
)

print("Starting CatBoost validation...", flush=True)

# Train on earlier sales, test on each future month.
run.validate()

print("Validation completed. Calculating scores...", flush=True)

# Display MdAPE and MAPE.
run.report()


Starting CatBoost validation...
2025-12: trained on 139,931 earlier sales, predicted 1,238 homes
2026-01: trained on 141,169 earlier sales, predicted 1,058 homes
2026-02: trained on 142,227 earlier sales, predicted 1,442 homes
2026-03: trained on 143,669 earlier sales, predicted 1,816 homes
2026-04: trained on 145,485 earlier sales, predicted 2,224 homes
2026-05: trained on 147,709 earlier sales, predicted 2,291 homes
Wrote outputs/validation_predictions.csv
Validation completed. Calculating scores...
# Validation report

Walk-forward validation, 2025-12 to 2026-05: each month was predicted by models trained only on the sales before it.
Errors are (predicted - sold) / sold. Brackets: bootstrap 95% interval.

| Track | Homes | MAPE | MdAPE | Median signed error |
|---|---|---|---|---|
| off_market | 10,069 | 13.73% (13.50 to 13.98) | 10.77% (10.55 to 11.02) | +1.50% |
| list_aware | 10,069 | 5.39% (5.31 to 5.49) | 4.30% (4.21 to 4.42) | +0.88% |

MAPE: mean absolute percentage error. Md

In [ ]:

# BACKUP: Save our original CatBoost results.
# This allows us to compare before and after changes.

from pathlib import Path
import shutil

source = Path("outputs/validation_predictions.csv")
backup = Path("outputs/catboost_original_validation.csv")

if source.exists():
    shutil.copy2(source, backup)
    print("Original predictions saved:", backup)
else:
    print("Validation file not found. Check the project folder.")


Original predictions saved: outputs/catboost_original_validation.csv


In [ ]:

# RELOAD: Apply our edited CatBoost model.
# This does not run validation yet.

import importlib
import pineapple_catboost
import models

importlib.reload(pineapple_catboost)

models.make_model = lambda track: (
    pineapple_catboost.CatBoostPriceModel(track)
)

# Check that the new feature exists.
test_features = pineapple_catboost.make_features(
    listings.head(),
    "off_market"
)

print("New feature included:",
      "community_property_type" in test_features.columns)

print("Example values:")
print(test_features["community_property_type"].head())

print("Model:", type(models.make_model("off_market")).__name__)


New feature included: True
Example values:
0       Sumacpark_Condo Apt
1    Hickorymills_Condo Apt
2      Hazelmills_Condo Apt
3       Sumacbank_Condo Apt
4        Ashworth_Condo Apt
Name: community_property_type, dtype: object
Model: CatBoostPriceModel


In [ ]:

# EXPERIMENT 2: Community × Property Type
#
# Keep the same model settings and validation dates.
# Only the feature engineering has changed.

import run

print("Validating community-property interaction...")
run.validate()

print("Calculating new accuracy...")
run.report()


Validating community-property interaction...
2025-12: trained on 139,931 earlier sales, predicted 1,238 homes
2026-01: trained on 141,169 earlier sales, predicted 1,058 homes
2026-02: trained on 142,227 earlier sales, predicted 1,442 homes
2026-03: trained on 143,669 earlier sales, predicted 1,816 homes
2026-04: trained on 145,485 earlier sales, predicted 2,224 homes
2026-05: trained on 147,709 earlier sales, predicted 2,291 homes
Wrote outputs/validation_predictions.csv
Calculating new accuracy...
# Validation report

Walk-forward validation, 2025-12 to 2026-05: each month was predicted by models trained only on the sales before it.
Errors are (predicted - sold) / sold. Brackets: bootstrap 95% interval.

| Track | Homes | MAPE | MdAPE | Median signed error |
|---|---|---|---|---|
| off_market | 10,069 | 13.66% (13.43 to 13.91) | 10.83% (10.57 to 11.10) | +1.40% |
| list_aware | 10,069 | 5.39% (5.31 to 5.48) | 4.30% (4.22 to 4.40) | +0.90% |

MAPE: mean absolute percentage error. MdAPE

In [ ]:

# COMPARE: Did the new feature improve accuracy?
# Lower MdAPE is better.

import pandas as pd

def score_predictions(path):
    df = pd.read_csv(path)
    df["APE"] = (
        (df["predicted_price"] - df["sold_price"]).abs()
        / df["sold_price"] * 100
    )
    return df.groupby("track")["APE"].median()

before = score_predictions(
    "outputs/catboost_original_validation.csv"
)

after = score_predictions(
    "outputs/validation_predictions.csv"
)

comparison = pd.DataFrame({
    "Before MdAPE": before,
    "After MdAPE": after
})

comparison["Change (pp)"] = (
    comparison["After MdAPE"]
    - comparison["Before MdAPE"]
)

display(comparison.round(3))


FileNotFoundError: [Errno 2] No such file or directory: 'outputs/catboost_original_validation.csv'

In [ ]:

# Save any previous validation results before replacing
# the working validation output with our experiment.

from pathlib import Path
import shutil

output = Path("outputs/validation_predictions.csv")
if output.exists():
    shutil.copy2(
        output,
        "outputs/validation_before_catboost.csv"
    )

# Train and validate CatBoost for both tracks.
run.validate()

# Generate official accuracy statistics.
run.report()


2025-12: trained on 139,931 earlier sales, predicted 1,238 homes
2026-01: trained on 141,169 earlier sales, predicted 1,058 homes
2026-02: trained on 142,227 earlier sales, predicted 1,442 homes
2026-03: trained on 143,669 earlier sales, predicted 1,816 homes
2026-04: trained on 145,485 earlier sales, predicted 2,224 homes
2026-05: trained on 147,709 earlier sales, predicted 2,291 homes
Wrote outputs/validation_predictions.csv
# Validation report

Walk-forward validation, 2025-12 to 2026-05: each month was predicted by models trained only on the sales before it.
Errors are (predicted - sold) / sold. Brackets: bootstrap 95% interval.

| Track | Homes | MAPE | MdAPE | Median signed error |
|---|---|---|---|---|
| off_market | 10,069 | 13.66% (13.43 to 13.91) | 10.83% (10.57 to 11.10) | +1.40% |
| list_aware | 10,069 | 5.39% (5.31 to 5.48) | 4.30% (4.22 to 4.40) | +0.90% |

MAPE: mean absolute percentage error. MdAPE: median absolute percentage error.
A median signed error above 0 means t

In [ ]:

# Inspect the validation predictions.
# This does NOT retrain the model.

import pandas as pd

results = pd.read_csv("outputs/validation_predictions.csv")

print("Columns:")
print(results.columns.tolist())

display(results.head())


Columns:
['ml_num', 'track', 'month', 'house_type_name', 'sold_price', 'predicted_price']


,ml_num,track,month,house_type_name,sold_price,predicted_price
0,SYN-0154350,off_market,2025-12,Condo Apt,797100.0,1127224.0
1,SYN-0150378,off_market,2025-12,Detached,1821500.0,1708560.0
2,SYN-0148591,off_market,2025-12,Condo Apt,711300.0,738683.0
3,SYN-0155236,off_market,2025-12,Condo Apt,257500.0,341559.0
4,SYN-0040814,off_market,2025-12,Condo Apt,516500.0,660372.0


In [ ]:

# EXPERIMENT: Analyze where CatBoost makes mistakes.
# This does not change or retrain the model.

import pandas as pd
import matplotlib.pyplot as plt

results = pd.read_csv("outputs/validation_predictions.csv")

# Signed error: positive = prediction too high.
results["signed_error"] = (
    (results["predicted_price"] - results["sold_price"])
    / results["sold_price"] * 100
)

# Absolute error: how far off, regardless of direction.
results["absolute_error"] = results["signed_error"].abs()

for track in ["off_market", "list_aware"]:

    df = results[results["track"] == track].copy()

    # Split properties into five groups by actual price.
    # Actual sale price is used ONLY for evaluation.
    df["price_group"] = pd.qcut(
        df["sold_price"].rank(method="first"),
        q=5,
        labels=[
            "Cheapest 20%", "Low 20%", "Middle 20%",
            "High 20%", "Most expensive 20%"
        ]
    )

    summary = df.groupby(
        "price_group", observed=True
    ).agg(
        homes=("ml_num", "count"),
        MdAPE=("absolute_error", "median"),
        median_bias=("signed_error", "median")
    ).round(2)

    print("\n", track.upper())
    display(summary)

    # Visualize systematic over/underprediction.
    plt.figure(figsize=(9, 4))
    summary["median_bias"].plot(kind="bar")
    plt.axhline(0, color="black", linewidth=1)
    plt.title(f"{track}: Median signed error by price group")
    plt.ylabel("Median signed error (%)")
    plt.xticks(rotation=25, ha="right")
    plt.tight_layout()
    plt.show()


In [ ]:

# Analyze prediction accuracy by property type.
# No model training occurs here.

for track in ["off_market", "list_aware"]:

    df = results[results["track"] == track]

    summary = df.groupby("house_type_name").agg(
        homes=("ml_num", "count"),
        MdAPE=("absolute_error", "median"),
        median_bias=("signed_error", "median")
    ).round(2)

    print("\n", track.upper(), "- Property Types")
    display(summary.sort_values("MdAPE", ascending=False))



 OFF_MARKET - Property Types


,homes,MdAPE,median_bias
house_type_name,,,
Co-Op Apt,48,17.54,0.81
Detached,3123,11.32,0.69
Semi-Detached,1034,11.07,-3.54
Condo Apt,4843,10.66,2.75
Freehold Townhouse,405,10.20,-1.92
Condo Townhouse,616,9.16,4.27



 LIST_AWARE - Property Types


,homes,MdAPE,median_bias
house_type_name,,,
Semi-Detached,1034,5.86,0.32
Freehold Townhouse,405,5.85,1.26
Detached,3123,5.43,1.24
Condo Townhouse,616,4.61,1.23
Co-Op Apt,48,4.15,0.49
Condo Apt,4843,3.53,0.73


In [ ]:

# EXPERIMENT: Community-level error analysis
# Goal: Find communities where CatBoost consistently
# predicts prices too high or too low.
#
# This is analysis only. It does NOT train a new model.

import pandas as pd

# Load the CatBoost validation predictions.
results = pd.read_csv(
    "outputs/validation_predictions.csv"
)

# Load historical housing data, including community names.
import data
listings = data.load("listings")

# Add community information to each validation prediction.
community_info = listings[
    ["ml_num", "community"]
].drop_duplicates("ml_num")

df = results.merge(
    community_info,
    on="ml_num",
    how="left",
    validate="many_to_one"
)

# Calculate prediction errors.
df["signed_error"] = (
    (df["predicted_price"] - df["sold_price"])
    / df["sold_price"] * 100
)

df["absolute_error"] = df["signed_error"].abs()

# Analyze off-market results first.
off = df[df["track"] == "off_market"]

summary = off.groupby("community").agg(
    homes=("ml_num", "count"),
    MdAPE=("absolute_error", "median"),
    median_bias=("signed_error", "median")
).round(2)

# Focus on communities with at least 50 homes.
summary = summary[summary["homes"] >= 50]

print("MOST DIFFICULT COMMUNITIES:")
display(
    summary.sort_values(
        "MdAPE", ascending=False
    ).head(10)
)

print("\nCOMMUNITIES WITH MOST OVERPREDICTION:")
display(
    summary.sort_values(
        "median_bias", ascending=False
    ).head(10)
)

print("\nCOMMUNITIES WITH MOST UNDERPREDICTION:")
display(
    summary.sort_values(
        "median_bias"
    ).head(10)
)


MOST DIFFICULT COMMUNITIES:


,homes,MdAPE,median_bias
community,,,
Cypressglen,54,21.85,19.08
Chestnutbank,100,21.47,20.80
Hazelfield,68,20.31,-12.60
Sumacwood,102,19.84,17.51
Sumacfield,52,18.11,2.89
Lindenglen,62,17.80,2.06
Willowpark,58,15.70,-8.87
Alderworth,54,15.45,12.27
Juniperheights,93,14.73,-3.04



COMMUNITIES WITH MOST OVERPREDICTION:


,homes,MdAPE,median_bias
community,,,
Chestnutbank,100,21.47,20.80
Cypressglen,54,21.85,19.08
Sumacwood,102,19.84,17.51
Alderworth,54,15.45,12.27
Oakbourne,122,13.39,12.04
Cedarbank,101,14.68,11.76
Sycamoreview,51,11.70,10.87
Poplarcrest,77,13.32,9.08
Tamarackmills,65,10.66,8.72



COMMUNITIES WITH MOST UNDERPREDICTION:


,homes,MdAPE,median_bias
community,,,
Hazelfield,68,20.31,-12.60
Magnoliaworth,80,12.68,-9.99
Magnoliacrest,58,11.00,-9.36
Birchhollow,69,10.36,-9.13
Maplegrove,54,11.62,-9.12
Willowpark,58,15.70,-8.87
Beechglen,59,13.52,-8.18
Sprucehaven,55,11.05,-8.10
Lindendale,101,12.75,-6.82


## 4. Predict and check

This fits the models on all 150,000 sales and writes the two submission files.

In [ ]:
run.predict()
run.check()

In [ ]:
# In Colab: download the two submission files.
if "google.colab" in sys.modules:
    from google.colab import files
    files.download("submissions/submission_off_market.csv")
    files.download("submissions/submission_list_aware.csv")